# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive template for loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing all data elements (record sets, fields, columns) **by their `@id`** as per the Croissant specification.

### Dataset Source
The dataset Croissant schema is provided at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the Croissant schema using `mlcroissant`.

> **Note:** All data references are made via their entity `@id`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Keep as an mlcroissant Metadata object

# Print dataset name and description
print(f"{getattr(metadata, 'name', '[Name not found]')}: {getattr(metadata, 'description', '[Description not found]')}")
# Optionally show available top-level keys
print("\nTop-level metadata attributes:")
print([attr for attr in dir(metadata) if not attr.startswith('_')])

## 2. Data Overview
Review available record sets, fields, and their `@id`s. 

All entities should be referenced by their `@id`, which you can retrieve programmatically. Let's enumerate the record sets and demonstrate their associated fields and columns.

In [ ]:
# List all available record sets by their @id.
record_sets = dataset.metadata.record_sets  # list of RecordSet objects
print("Available Record Sets and Fields (by @id):\n")
all_recordset_ids = []
for rs in record_sets:
    print(f"RecordSet @id: {rs.id} - name: {getattr(rs, 'name', '')}")
    all_recordset_ids.append(rs.id)
    # List fields
    for field in rs.fields:
        print(f"    Field @id: {field.id} - name: {getattr(field, 'name', '')}")
        # List column (source) if available
        if hasattr(field, 'column') and field.column is not None:
            print(f"        Column @id: {field.column.id}")
    print()
# Example: Print first few records for the first record set
if len(record_sets) > 0:
    example_recordset_id = record_sets[0].id
    print(f"\nSample records from RecordSet {example_recordset_id}:")
    for count, rec in enumerate(dataset.records(record_set=example_recordset_id)):
        print(json.dumps(rec, indent=2))
        if count >= 1:
            break


## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis.

We use the record set `@id`s from the previous step as dynamic variables for programmatic loading.


In [ ]:
# Extract all data to DataFrames, keys by record set @id
dataframes = {}
for rs in record_sets:
    rs_id = rs.id
    # Load all records for this record set
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for RecordSet {rs_id}")
# Show available columns for the main record set
if len(record_sets) > 0:
    main_rs_id = record_sets[0].id
    print(f"\nColumns in DataFrame for RecordSet {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    # Show first few rows
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric field and a grouping field (all by their `@id`) found in the record set and demonstrate filtering, normalization, and grouping. 

Replace the `numeric_field_id` and `group_field_id` below with the relevant `@id` values obtained from the overview.

In [ ]:
import numpy as np

# Select main record set and choose a numeric and groupable field by @id
record_set_id = main_rs_id
df = dataframes[record_set_id]

# For demonstration, automatically try to select likely numeric and group fields by column name.
# (Change these if schema specifies different columns by @id.)
numeric_candidates = [c for c in df.columns if df[c].dtype in [np.float64, np.int64, 'float64', 'int64']]
if not numeric_candidates:
    # Try columns that contain 'abundance', 'MW' or 'count' in their name
    numeric_candidates = [c for c in df.columns if any(x in c.lower() for x in ['abundance', 'mw', 'count', 'coverage', 'peptide'])]
group_candidates = [c for c in df.columns if 'sample' in c.lower() or 'group' in c.lower() or 'accession' in c.lower()]

# Display suggestions
print(f"Numeric field candidates: {numeric_candidates}")
print(f"Group field candidates: {group_candidates}")

# Replace these with actual @id values from overview for production use
numeric_field = numeric_candidates[0] if numeric_candidates else None
group_field = group_candidates[0] if group_candidates else None
print(f"Using numeric_field (as @id): {numeric_field}")
print(f"Using group_field (as @id): {group_field}")

# Continue only if fields were found
if numeric_field and numeric_field in df.columns:
    # Convert values to numeric if not already
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean() if not pd.isna(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} rows")
    print(filtered_df.head())
    
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution and relationships between fields in your main record set. For demonstration, we'll plot the filtered and normalized numeric field by group.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(10, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- The FAIR² mass spectrometry dataset provides detailed protein measurements and annotations from human mast cell vesicles.
- Record sets, fields, and columns are referenced by their `@id` according to the Croissant standard to ensure reproducibility.
- Using `mlcroissant`, we extracted, filtered, normalized, and visualized the data for common analysis tasks.
- You should adapt the field `@id` variables and EDA operations for your domain-specific questions and hypotheses for deeper insights.
